# 01 — Exploratory Data Analysis

The dataset is a static two-day snapshot of European card transactions from September 2013.
Features V1–V28 are PCA-transformed and anonymized; we cannot recover the original business variables.
The goal here is to understand class imbalance, feature separation, and time patterns before modeling.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from fraud.data import load, FIG

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

df = load()
V = [f"V{i}" for i in range(1, 29)]
print("Shape:", df.shape)
print(df["Class"].value_counts())
print(f"Fraud rate: {df['Class'].mean():.4%}")
print(f"Duplicate rows: {df.duplicated().sum()}")

## Class imbalance

492 frauds out of 284,807 transactions — roughly 0.17%. Any model that predicts "legitimate" for
every transaction achieves 99.83% accuracy. That is why we use PR-AUC as the primary metric.

The dataset has about 1,000 exact duplicate rows. We keep them because the time-based split
prevents leakage and they may represent real repeated charges.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for c, lab, col in [(0, "legitimate", "#4878CF"), (1, "fraud", "#D65F5F")]:
    ax.hist(
        np.log10(df.loc[df.Class == c, "Amount"] + 1),
        bins=50, alpha=0.6, density=True, label=lab, color=col
    )
ax.set_xlabel("log10(Amount + 1), USD")
ax.set_ylabel("density")
ax.set_title("Transaction amount by class")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "eda_amount.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df["hour"] = (df.Time // 3600).astype(int)
rate = df.groupby("hour")["Class"].agg(["mean", "sum", "size"])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rate.index, rate["mean"].mul(100), marker="o", ms=4, color="#4878CF")
ax.set_xlabel("hours since first transaction in dataset")
ax.set_ylabel("fraud rate (%)")
ax.set_title("Fraud rate over the two-day window")
fig.tight_layout()
fig.savefig(FIG / "eda_fraud_rate_time.png", dpi=150, bbox_inches="tight")
plt.show()
print("Hourly fraud counts (sum):")
print(rate["sum"].describe())

In [ ]:
ks = pd.Series(
    {c: ks_2samp(df.loc[df.Class == 1, c], df.loc[df.Class == 0, c]).statistic
     for c in V}
).sort_values()

fig, ax = plt.subplots(figsize=(6, 8))
ks.plot.barh(ax=ax, color="#4878CF")
ax.set_xlabel("KS statistic (fraud vs legitimate)")
ax.set_title("Feature separation by class\n(higher = more separating)")
fig.tight_layout()
fig.savefig(FIG / "eda_ks_features.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nTop 5 most separating features:")
print(ks.tail(5))

In [ ]:
print("Amount by class:")
print(df.groupby("Class")["Amount"].describe().round(2))
print("\nMedian amount — legitimate:", df.loc[df.Class==0,"Amount"].median())
print("Median amount — fraud:", df.loc[df.Class==1,"Amount"].median())

## What this means for modeling

1. **Imbalance**: PR-AUC is the right metric. Accuracy is misleading.
2. **Time structure**: Fraud rate varies over the two-day window. We use a time-based train/val/test
   split (70/15/15) to simulate a production deployment.
3. **Feature anonymization**: V1–V28 are PCA components. We can identify which are predictive but
   cannot interpret them in business terms.
4. **Amount**: Fraud transactions cover a wide range; small frauds appear alongside large ones.
   The cost function (Section 3) uses the actual amount as the false-negative cost.